# Chroma (2026 업데이트판)

Chroma는 개발자 생산성에 초점을 맞춘 AI 네이티브 오픈소스 벡터 데이터베이스입니다(Apache 2.0). 로컬 메모리/디스크, 자체 호스팅 서버, Chroma Cloud 어디에나 같은 API로 붙습니다.

### 원본 대비 변경 사항
| 항목 | 원본 | 현재 권장 |
|---|---|---|
| LangSmith 설정 | `langchain_teddynote.logging` | 환경 변수 (`LANGSMITH_*`) |
| 문서 로더 | `langchain_community.document_loaders.TextLoader` | 파이썬 기본 파일 읽기 + `create_documents()` |
| 문서 분할 | `langchain.text_splitter` + `loader.load_and_split()` | `langchain_text_splitters` (`load_and_split()` 은 deprecated) |
| 임베딩 모델 | `OpenAIEmbeddings()` (기본값) | `OpenAIEmbeddings(model="text-embedding-3-small")` 명시 |
| 거리 함수 지정 | `collection_metadata={"hnsw:space": "cosine"}` | `collection_configuration={"hnsw": {"space": "cosine"}}` |
| 패키지 | `langchain-chroma` 0.1.x / `chromadb` 0.5.x | `langchain-chroma` 1.x / `chromadb` 1.3.5+ |
| ID로 조회 | `db.get(ids)` (Chroma 전용 반환 형식) | `db.get_by_ids(ids)` (표준 `VectorStore` 인터페이스, `Document` 반환) |
| 멀티모달 임베딩 | `langchain_experimental.open_clip.OpenCLIPEmbeddings` | `open_clip` 을 직접 감싼 `Embeddings` 구현 |
| 이미지 캡션 생성 | `langchain_teddynote.models.MultiModal` | `init_chat_model` + 표준 이미지 content block |

**`langchain-community` / `langchain-experimental` 에 대해**: 두 패키지 모두 2026년에 지원 종료(sunset)되었습니다. 설치는 되지만 더 이상 수정되지 않으므로, 새 코드에서는 `langchain-core` 또는 공급자별 전용 패키지(`langchain-chroma`, `langchain-openai` 등)를 사용합니다.

**참고 링크**
- [Chroma LangChain 통합 문서](https://docs.langchain.com/oss/python/integrations/vectorstores/chroma)
- [Chroma 공식 문서](https://docs.trychroma.com)
- [LangChain 지원 VectorStore 목록](https://docs.langchain.com/oss/python/integrations/vectorstores)

In [ ]:
%pip install -qU langchain-chroma langchain-openai langchain-text-splitters python-dotenv

## 환경 설정

- `.env` 파일의 API 키를 `python-dotenv` 로 불러옵니다.
- **변경점**: 책에서 사용한 `langchain_teddynote.logging.langsmith()` 는 서드파티 헬퍼입니다. 현재 LangSmith 공식 방식은 환경 변수(`LANGSMITH_TRACING`, `LANGSMITH_API_KEY`, `LANGSMITH_PROJECT`)만 설정하는 것이며, 별도 패키지가 필요 없습니다.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # .env 파일의 키를 환경 변수로 로드

# LangSmith 추적 (LANGSMITH_API_KEY 는 .env 에 넣어 둡니다)
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "CH09-VectorStore")

## 샘플 데이터 로드

**변경점**
- `TextLoader`(`langchain_community`)는 파일을 읽어 `Document` 로 감싸는 일만 하므로, 파이썬 기본 기능으로 읽고 `create_documents()` 로 바로 분할합니다. 출처 정보는 `metadatas` 로 직접 넣습니다.
- `loader.load_and_split()` 는 `langchain-core` 에서 "deprecated 로 간주한다"고 명시된 메서드입니다. 분할기는 `split_documents()` / `create_documents()` 로 직접 호출합니다.
- 텍스트 분할기는 `langchain.text_splitter` 가 아니라 독립 패키지 `langchain_text_splitters` 에서 가져옵니다.

In [ ]:
from pathlib import Path

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=0)


def load_and_split(path: str):
    # 텍스트 파일을 읽어 source 메타데이터를 붙인 청크 리스트로 변환합니다.
    raw_text = Path(path).read_text(encoding="utf-8")
    return text_splitter.create_documents([raw_text], metadatas=[{"source": path}])


split_doc1 = load_and_split("data/nlp-keywords.txt")
split_doc2 = load_and_split("data/finance-keywords.txt")

len(split_doc1), len(split_doc2)

## 임베딩 모델

**변경점**: `OpenAIEmbeddings()` 를 인자 없이 만들면 레거시 모델인 `text-embedding-ada-002` 가 선택됩니다. 항상 모델명을 명시하세요.

In [ ]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## VectorStore 생성

### 벡터 저장소 생성 (`from_documents`)

`from_documents` 클래스 메서드는 `Document` 리스트로부터 벡터 저장소를 생성합니다.

**주요 매개변수** (langchain-chroma 1.x 기준)

- `documents` (list[Document]): 저장할 문서 리스트
- `embedding` (Embeddings | None): 임베딩 모델
- `ids` (list[str] | None): 문서 ID 리스트. 없으면 UUID 자동 생성
- `collection_name` (str): 컬렉션 이름 (`namespace` 역할)
- `persist_directory` (str | None): 저장 디렉토리. 지정하지 않으면 메모리에만 존재
- `collection_configuration` (CreateCollectionConfiguration | None): **신규** — 인덱스 설정. 거리 함수는 여기서 지정합니다 (`{"hnsw": {"space": "cosine"}}`)
- `collection_metadata` (dict | None): 컬렉션에 붙일 임의의 메타데이터
- `host` / `port` / `ssl` / `headers`: **신규** — 원격 Chroma 서버에 접속할 때 사용
- `chroma_cloud_api_key` / `tenant` / `database`: **신규** — Chroma Cloud 접속용
- `client` / `client_settings`: 직접 만든 `chromadb` 클라이언트/설정을 넘길 때 사용

**변경점**: 거리 함수를 `collection_metadata={"hnsw:space": "cosine"}` 로 지정하던 방식은 `collection_configuration={"hnsw": {"space": "cosine"}}` 로 바뀌었습니다. Chroma의 기본 거리는 제곱 L2(`l2`)인데 OpenAI 임베딩은 코사인 유사도를 전제로 하므로 `cosine` 을 지정하는 편이 좋습니다. 뒤에서 쓰는 `similarity_score_threshold` 검색도 이 설정에 따라 점수 변환 방식이 달라집니다.

In [ ]:
from langchain_chroma import Chroma

# 거리 함수를 cosine 으로 지정 (기본값은 제곱 L2)
HNSW_COSINE = {"hnsw": {"space": "cosine"}}

# DB 생성 (persist_directory 미지정 -> 메모리에만 존재)
db = Chroma.from_documents(
    documents=split_doc1,
    embedding=embeddings,
    collection_name="my_db",
    collection_configuration=HNSW_COSINE,
)

`persist_directory` 를 지정하면 디스크에 파일 형태로 저장됩니다.

**변경점**: chromadb 0.4.x 시절의 `db.persist()` 호출은 더 이상 필요하지 않습니다(1.x 에서 제거되었습니다). `persist_directory` 를 지정하면 쓰기 시점에 자동으로 디스크에 반영됩니다.

In [ ]:
# 저장할 경로 지정
DB_PATH = "./chroma_db"

persist_db = Chroma.from_documents(
    documents=split_doc1,
    embedding=embeddings,
    persist_directory=DB_PATH,
    collection_name="my_db",
    collection_configuration=HNSW_COSINE,
)

저장된 데이터를 다시 불러옵니다. 이때는 생성자에 `embedding_function` 으로 임베딩 모델을 넘깁니다(`from_documents` 의 `embedding` 과 인자 이름이 다릅니다).

In [ ]:
# 디스크에서 컬렉션을 로드
persist_db = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings,
    collection_name="my_db",
)

# 저장된 데이터 확인 (ids / documents / metadatas 반환)
persist_db.get(limit=2)

`collection_name` 을 다르게 지정하면 데이터가 없는 새 컬렉션이 만들어지므로 아무 결과도 나오지 않습니다.

In [ ]:
persist_db2 = Chroma(
    persist_directory=DB_PATH,
    embedding_function=embeddings,
    collection_name="my_db2",
)

persist_db2.get()

### 벡터 저장소 생성 (`from_texts`)

문자열 리스트로부터 바로 생성합니다. `metadatas`, `ids` 를 함께 넘길 수 있고 `ids` 를 생략하면 UUID가 자동 부여됩니다.

In [ ]:
db2 = Chroma.from_texts(
    ["안녕하세요. 정말 반갑습니다.", "제 이름은 테디입니다."],
    embedding=embeddings,
    collection_configuration=HNSW_COSINE,
)

db2.get()

## 유사도 검색

`similarity_search(query, k=4, filter=None)` 은 쿼리와 가장 유사한 문서를 반환합니다. 점수까지 필요하면 `similarity_search_with_score`, 이미 계산한 벡터로 찾으려면 `similarity_search_by_vector` 를 사용합니다.

In [ ]:
db.similarity_search("TF IDF 에 대하여 알려줘")

`k` 값으로 반환 개수를 지정합니다.

In [ ]:
db.similarity_search("TF IDF 에 대하여 알려줘", k=2)

`filter` 에 메타데이터 조건을 넘겨 검색 범위를 좁힐 수 있습니다.

Chroma가 지원하는 연산자:
- 비교: `$eq`, `$ne`, `$gt`, `$gte`, `$lt`, `$lte`
- 논리: `$and`, `$or`
- 포함: `$in`, `$nin`
- 배열: `$contains`, `$not_contains`

`{"source": "data/nlp-keywords.txt"}` 처럼 값을 바로 쓰면 `{"source": {"$eq": "..."}}` 와 같은 의미입니다.

In [ ]:
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/nlp-keywords.txt"}, k=2
)

In [ ]:
# 다른 source 로 필터링 (db 에는 nlp 문서만 있으므로 결과가 비어 있습니다)
db.similarity_search(
    "TF IDF 에 대하여 알려줘", filter={"source": "data/finance-keywords.txt"}, k=2
)

### 점수와 함께 검색

`similarity_search_with_score` 는 `(Document, distance)` 튜플을 반환합니다. 여기서 값은 **거리**이므로 작을수록 유사합니다. 반대로 `similarity_search_with_relevance_scores` 는 0~1 로 정규화된 **유사도**를 반환하며, 이 변환은 앞에서 지정한 `space`(여기서는 `cosine`)에 따라 결정됩니다.

In [ ]:
for doc, distance in db.similarity_search_with_score("TF IDF 에 대하여 알려줘", k=2):
    print(f"[distance={distance:.4f}] {doc.page_content[:80]}")

print()

for doc, score in db.similarity_search_with_relevance_scores(
    "TF IDF 에 대하여 알려줘", k=2
):
    print(f"[relevance={score:.4f}] {doc.page_content[:80]}")

## 문서 추가 / 수정 / 삭제

`add_documents(documents, ids=None)` 는 문서를 추가하거나 같은 ID면 덮어씁니다(upsert).

- 문서의 `id` 필드를 지정하거나 `ids` 인자로 넘길 수 있고, `ids` 가 우선합니다.
- `ids` 개수와 문서 개수가 다르면 `ValueError` 가 발생합니다.

In [ ]:
from langchain_core.documents import Document

db.add_documents(
    [
        Document(
            page_content="안녕하세요! 이번엔 도큐먼트를 새로 추가해 볼께요",
            metadata={"source": "mydata.txt"},
            id="1",
        )
    ]
)

**변경점**: ID로 문서를 가져올 때는 Chroma 전용 `get()` 대신 표준 `VectorStore` 인터페이스인 `get_by_ids()` 를 쓰는 편이 좋습니다. 다른 벡터 저장소로 갈아타도 코드가 그대로 동작하고, 결과가 딕셔너리가 아니라 `Document` 리스트로 돌아옵니다.

In [ ]:
# 표준 인터페이스 (권장)
db.get_by_ids(["1"])

In [ ]:
# Chroma 전용 조회 (ids / documents / metadatas 딕셔너리 반환)
db.get("1")

`add_texts` 는 텍스트를 임베딩해 바로 추가합니다. 기존 ID를 다시 쓰면 upsert 되어 기존 문서가 대체됩니다.

In [ ]:
db.add_texts(
    ["이전에 추가한 Document 를 덮어쓰겠습니다.", "덮어쓴 결과가 어떤가요?"],
    metadatas=[{"source": "mydata.txt"}, {"source": "mydata.txt"}],
    ids=["1", "2"],
)

db.get_by_ids(["1"])

### 문서 수정 (`update_documents`)

ID를 유지한 채 내용/메타데이터만 바꾸고 싶다면 `update_documents(ids, documents)` 를 사용합니다.

In [ ]:
db.update_documents(
    ids=["2"],
    documents=[
        Document(page_content="수정된 내용입니다.", metadata={"source": "mydata.txt"})
    ],
)

db.get_by_ids(["2"])

### 문서 삭제 (`delete`)

`delete(ids)` 로 지정한 ID의 문서를 삭제합니다. `ids` 가 `None` 이면 아무 것도 하지 않습니다.

In [ ]:
db.delete(ids=["1"])

db.get_by_ids(["1", "2"])

In [ ]:
# where 조건(메타데이터)으로 조회
db.get(where={"source": "mydata.txt"})

### 컬렉션 초기화 (`reset_collection`)

컬렉션을 비웁니다. 컬렉션 자체를 삭제하려면 `delete_collection()` 을 사용합니다.

In [ ]:
db.reset_collection()
db.get()

## 벡터 저장소를 검색기(Retriever)로 변환

`as_retriever()` 는 벡터 저장소를 `VectorStoreRetriever`(Runnable)로 바꿔 줍니다. 체인/에이전트에 그대로 꽂을 수 있습니다.

**매개변수**
- `search_type`: `"similarity"`(기본), `"mmr"`, `"similarity_score_threshold"`
- `search_kwargs`:
  - `k`: 반환할 문서 수 (기본값 4)
  - `score_threshold`: 최소 유사도 임계값 (`similarity_score_threshold` 전용)
  - `fetch_k`: MMR 후보로 가져올 문서 수 (기본값 20)
  - `lambda_mult`: MMR 다양성 조절 (0~1, 기본값 0.5. 0에 가까울수록 다양성 우선)
  - `filter`: 메타데이터 필터

In [ ]:
# 두 문서 집합을 모두 담은 DB 생성
db = Chroma.from_documents(
    documents=split_doc1 + split_doc2,
    embedding=embeddings,
    collection_name="nlp",
    collection_configuration=HNSW_COSINE,
)

In [ ]:
# 기본 검색기: 유사도 상위 4개
retriever = db.as_retriever()
retriever.invoke("Word2Vec 에 대하여 알려줘")

MMR(Maximal Marginal Relevance)로 다양성을 높여 검색합니다.

In [ ]:
retriever = db.as_retriever(
    search_type="mmr", search_kwargs={"k": 6, "lambda_mult": 0.25, "fetch_k": 10}
)
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 후보는 10개를 보되 최종 2개만 반환
retriever = db.as_retriever(search_type="mmr", search_kwargs={"k": 2, "fetch_k": 10})
retriever.invoke("Word2Vec 에 대하여 알려줘")

임계값 이상의 유사도를 가진 문서만 반환합니다. 점수는 앞에서 지정한 `space`(cosine) 기준으로 0~1 로 정규화된 값입니다.

**변경점**: 원본의 `score_threshold=0.8` 은 거리 함수를 지정하지 않은 상태에서 정한 값이라 결과가 비기 쉽습니다. 임계값은 위 `similarity_search_with_relevance_scores` 로 실제 점수 분포를 확인한 뒤 정하세요.

In [ ]:
retriever = db.as_retriever(
    search_type="similarity_score_threshold", search_kwargs={"score_threshold": 0.5}
)
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 가장 유사한 단일 문서만 검색
retriever = db.as_retriever(search_kwargs={"k": 1})
retriever.invoke("Word2Vec 에 대하여 알려줘")

In [ ]:
# 특정 메타데이터 필터 적용
retriever = db.as_retriever(
    search_kwargs={"filter": {"source": "data/finance-keywords.txt"}, "k": 2}
)
retriever.invoke("ESG 에 대하여 알려줘")

## langchain-chroma 1.x 에서 추가된 기능

아래는 원본 집필 이후 추가된 기능들입니다. 별도 설정이 필요해 실행 셀로 넣지 않았지만, 실무에서 자주 쓰이므로 알아 두면 좋습니다.

**1) 하이브리드 검색 (`hybrid_search`)** — 조밀(dense) 벡터와 희소(sparse) 벡터 결과를 RRF(Reciprocal Rank Fusion)로 합칩니다. 컬렉션 스키마에 희소 벡터 인덱스(BM25 또는 SPLADE)를 먼저 설정해야 하며, 그 설정은 `chromadb` 클라이언트로 직접 컬렉션을 만든 뒤 `Chroma(client=...)` 로 넘기는 방식으로 합니다.

```python
from chromadb import Search, K, Knn, Rrf

search = (
    Search()
    .where(K("source") == "data/nlp-keywords.txt")
    .rank(
        Rrf(
            ranks=[
                Knn(query="Word2Vec", return_rank=True),
                Knn(query="Word2Vec", key="sparse_embedding", return_rank=True),
            ],
            weights=[2.0, 1.0],  # dense 가중치를 2배로
            k=60,
        )
    )
    .limit(5)
    .select(K.DOCUMENT, K.SCORE)
)
docs = db.hybrid_search(search)
```

**2) 컬렉션 복제 (`fork`)** — 기존 컬렉션을 복사해 새 이름으로 만듭니다. 인덱스를 다시 만들지 않으므로 A/B 테스트나 스냅샷 용도로 유용합니다.

```python
forked = db.fork("nlp-experiment")
```

**3) 원격 / Chroma Cloud 접속** — 같은 `Chroma` 클래스에 접속 정보만 다르게 넘깁니다.

```python
# 자체 호스팅 서버
Chroma(collection_name="nlp", embedding_function=embeddings, host="localhost", port=8000)

# Chroma Cloud
Chroma(
    collection_name="nlp",
    embedding_function=embeddings,
    chroma_cloud_api_key=os.environ["CHROMA_API_KEY"],
    tenant="<tenant-id>",
    database="<database>",
)
```

참고: [Chroma 하이브리드 검색 문서](https://docs.trychroma.com/cloud/search-api/hybrid-search)

---

## 멀티모달 검색

Chroma는 이미지와 텍스트를 같은 공간에 임베딩해 함께 검색하는 멀티모달 컬렉션을 지원합니다.

**변경점**
- 원본은 `langchain_experimental.open_clip.OpenCLIPEmbeddings` 를 사용했지만 **`langchain-experimental` 은 지원 종료(sunset)** 되었습니다. 아래에서는 `open_clip` 을 직접 감싼 최소 구현을 사용합니다. `Chroma.add_images()` / `similarity_search_by_image()` 는 임베딩 객체에 `embed_image(uris=[...])` 메서드만 있으면 동작하므로 30줄이면 충분합니다.
- 이미지 캡션 생성에 쓰인 `langchain_teddynote.models.MultiModal` 은 `init_chat_model` + LangChain 표준 이미지 content block 으로 대체했습니다.
- 기본 모델을 `ViT-H-14-378-quickgelu`(약 4GB) 대신 `ViT-B-32`(약 600MB)로 낮췄습니다. 품질이 필요하면 아래 벤치마크 표를 보고 바꾸세요.

In [ ]:
%pip install -qU open_clip_torch torch torchvision pillow datasets matplotlib pandas

### 데이터 세트

허깅페이스에 호스팅된 [COCO object detection dataset](https://huggingface.co/datasets/detection-datasets/coco) 의 일부만 내려받아 사용합니다.

In [ ]:
import os

from datasets import load_dataset
from matplotlib import pyplot as plt

# COCO 데이터셋을 스트리밍으로 로드 (전체를 내려받지 않습니다)
dataset = load_dataset(
    path="detection-datasets/coco", name="default", split="train", streaming=True
)

IMAGE_FOLDER = "tmp"
N_IMAGES = 20

plot_cols = 5
plot_rows = N_IMAGES // plot_cols
fig, axes = plt.subplots(plot_rows, plot_cols, figsize=(plot_cols * 2, plot_rows * 2))
axes = axes.flatten()

os.makedirs(IMAGE_FOLDER, exist_ok=True)
dataset_iter = iter(dataset)

for i in range(N_IMAGES):
    data = next(dataset_iter)
    image = data["image"]
    label = data["objects"]["category"][0]  # 첫 번째 객체의 카테고리를 레이블로 사용

    axes[i].imshow(image)
    axes[i].set_title(label, fontsize=8)
    axes[i].axis("off")

    image.convert("RGB").save(f"{IMAGE_FOLDER}/{i}.jpg")

plt.tight_layout()
plt.show()

### OpenCLIP 임베딩

[OpenCLIP](https://github.com/mlfoundations/open_clip) 은 이미지와 텍스트를 같은 벡터 공간으로 보내는 CLIP 계열 모델의 오픈소스 구현입니다.

**모델 벤치마크** (`model_name` / `checkpoint` 조합)

| Model | Training data (checkpoint) | Resolution | ImageNet zero-shot acc. |
|---|---|---|---|
| ViT-B-32 | LAION-2B (`laion2b_s34b_b79k`) | 224px | 66.6% |
| ViT-B-16 | DataComp-1B (`datacomp_xl_s13b_b90k`) | 224px | 73.5% |
| ViT-L-14 | LAION-2B (`laion2b_s32b_b82k`) | 224px | 75.3% |
| ViT-H-14 | LAION-2B (`laion2b_s32b_b79k`) | 224px | 78.0% |
| ViT-L-14 | DataComp-1B (`datacomp_xl_s13b_b90k`) | 224px | 79.2% |
| ViT-H-14-378-quickgelu | DFN-5B (`dfn5b`) | 378px | 84.4% |

사용 가능한 조합은 `open_clip.list_pretrained()` 로 확인할 수 있습니다.

In [ ]:
import open_clip
import pandas as pd

# 사용 가능한 모델/checkpoint 조합 확인
pd.DataFrame(open_clip.list_pretrained(), columns=["model_name", "checkpoint"]).head(10)

아래는 `open_clip` 을 LangChain `Embeddings` 인터페이스로 감싼 최소 구현입니다. `Chroma` 가 이미지를 다루기 위해 필요한 것은 `embed_documents` / `embed_query` / `embed_image` 세 개뿐입니다.

In [ ]:
from typing import List

import open_clip
import torch
from langchain_core.embeddings import Embeddings
from PIL import Image


class OpenCLIPEmbeddings(Embeddings):
    # open_clip 모델을 LangChain Embeddings 인터페이스로 감싼 구현.
    # langchain-experimental 이 sunset 되어 직접 감싸 사용합니다.
    # 이미지와 텍스트를 같은 벡터 공간으로 보내므로 상호 검색이 가능합니다.

    def __init__(
        self,
        model_name: str = "ViT-B-32",
        checkpoint: str = "laion2b_s34b_b79k",
    ) -> None:
        self.model, _, self.preprocess = open_clip.create_model_and_transforms(
            model_name, pretrained=checkpoint
        )
        self.tokenizer = open_clip.get_tokenizer(model_name)
        self.model.eval()

    @staticmethod
    def _to_list(features) -> List[List[float]]:
        # 코사인 유사도를 쓰기 위해 L2 정규화
        features = features / features.norm(dim=-1, keepdim=True)
        return features.cpu().numpy().tolist()

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        tokens = self.tokenizer(texts)
        with torch.no_grad():
            features = self.model.encode_text(tokens)
        return self._to_list(features)

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

    def embed_image(self, uris: List[str]) -> List[List[float]]:
        images = [self.preprocess(Image.open(uri).convert("RGB")) for uri in uris]
        with torch.no_grad():
            features = self.model.encode_image(torch.stack(images))
        return self._to_list(features)


image_embedding_function = OpenCLIPEmbeddings()

In [ ]:
# 이미지 경로를 리스트로 정리
image_uris = sorted(
    os.path.join(IMAGE_FOLDER, name)
    for name in os.listdir(IMAGE_FOLDER)
    if name.endswith(".jpg")
)

image_uris

### 이미지 설명 생성

**변경점**: 원본의 `langchain_teddynote.models.MultiModal` 대신 LangChain 표준 방식을 씁니다. 이미지는 `{"type": "image", "base64": ..., "mime_type": ...}` 형태의 content block 으로 전달하며, 이 형식은 공급자에 관계없이 동일합니다.

In [ ]:
import base64

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

MODEL_ID = os.getenv("CHAT_MODEL", "openai:gpt-5.4-mini")
vision_model = init_chat_model(MODEL_ID)

SYSTEM_PROMPT = "Your mission is to describe the image in detail."
USER_PROMPT = "Description should be written in one sentence (less than 60 characters)."


def describe_image(uri: str) -> str:
    # 이미지 파일 하나를 한 문장으로 설명합니다.
    image_b64 = base64.b64encode(Path(uri).read_bytes()).decode("utf-8")
    message = HumanMessage(
        content=[
            {"type": "text", "text": USER_PROMPT},
            {"type": "image", "base64": image_b64, "mime_type": "image/jpeg"},
        ]
    )
    response = vision_model.invoke([SystemMessage(SYSTEM_PROMPT), message])
    return response.text()


describe_image(image_uris[0])

In [ ]:
# 전체 이미지에 대한 설명 생성
descriptions = {uri: describe_image(uri) for uri in image_uris}
descriptions

In [ ]:
original_images = []
texts = []

plt.figure(figsize=(20, 10))

for i, image_uri in enumerate(image_uris):
    image = Image.open(image_uri).convert("RGB")

    plt.subplot(4, 5, i + 1)
    plt.imshow(image)
    plt.title(f"{os.path.basename(image_uri)}\n{descriptions[image_uri]}", fontsize=8)
    plt.xticks([])
    plt.yticks([])

    original_images.append(image)
    texts.append(descriptions[image_uri])

plt.tight_layout()

생성한 설명(텍스트)과 원본 이미지 사이의 코사인 유사도를 계산합니다. `embed_image` / `embed_documents` 가 이미 L2 정규화된 벡터를 돌려주므로 행렬곱 한 번이면 코사인 유사도가 됩니다.

In [ ]:
import numpy as np

img_features = np.array(image_embedding_function.embed_image(image_uris))
text_features = np.array(
    image_embedding_function.embed_documents(["This is " + desc for desc in texts])
)

similarity = text_features @ img_features.T
similarity.shape

In [ ]:
count = len(descriptions)
plt.figure(figsize=(20, 14))

plt.imshow(similarity, vmin=0.1, vmax=0.3, cmap="coolwarm")
plt.colorbar()

plt.yticks(range(count), texts, fontsize=18)
plt.xticks([])

# 원본 이미지를 x축 아래에 표시
for i, image in enumerate(original_images):
    plt.imshow(image, extent=(i - 0.5, i + 0.5, -1.6, -0.6), origin="lower")

for x in range(similarity.shape[1]):
    for y in range(similarity.shape[0]):
        plt.text(x, y, f"{similarity[y, x]:.2f}", ha="center", va="center", size=12)

for side in ["left", "top", "right", "bottom"]:
    plt.gca().spines[side].set_visible(False)

plt.xlim([-0.5, count - 0.5])
plt.ylim([count + 0.5, -2])
plt.title("Cosine Similarity", size=20)

### 멀티모달 VectorStore 생성

`Chroma.add_images(uris=[...])` 는 각 이미지를 base64 문자열로 저장하고, 임베딩 함수의 `embed_image()` 로 벡터를 만듭니다. 같은 컬렉션을 텍스트 쿼리로 검색하면 `embed_query()` 가 호출되므로 텍스트 → 이미지 검색이 가능합니다.

In [ ]:
image_db = Chroma(
    collection_name="multimodal",
    embedding_function=image_embedding_function,
    collection_configuration=HNSW_COSINE,
)

image_db.add_images(uris=image_uris)

검색 결과(base64 이미지)를 노트북에 렌더링하기 위한 헬퍼입니다.

**변경점**: `from langchain.schema import Document` 는 `from langchain_core.documents import Document` 로 바뀌었습니다.

In [ ]:
import io

from IPython.display import HTML, display


class ImageRetriever:
    # 검색된 base64 이미지를 노트북에 바로 표시하는 래퍼.

    def __init__(self, retriever):
        self.retriever = retriever

    def invoke(self, query: str):
        docs = self.retriever.invoke(query)
        if docs and isinstance(docs[0], Document):
            self.plt_img_base64(docs[0].page_content)
        else:
            print("검색된 이미지가 없습니다.")
        return docs

    @staticmethod
    def resize_base64_image(base64_string: str, size=(224, 224)) -> str:
        img = Image.open(io.BytesIO(base64.b64decode(base64_string)))
        resized_img = img.resize(size, Image.LANCZOS)
        buffered = io.BytesIO()
        resized_img.save(buffered, format=img.format or "JPEG")
        return base64.b64encode(buffered.getvalue()).decode("utf-8")

    @staticmethod
    def plt_img_base64(img_base64: str) -> None:
        display(HTML(f'<img src="data:image/jpeg;base64,{img_base64}" />'))

In [ ]:
retriever = image_db.as_retriever(search_kwargs={"k": 3})
image_retriever = ImageRetriever(retriever)

result = image_retriever.invoke("A Dog on the street")

In [ ]:
result = image_retriever.invoke("Motorcycle with a man")

이미지로 이미지를 검색할 수도 있습니다. `similarity_search_by_image(uri)` 는 이미지를 임베딩해 가장 비슷한 이미지를 찾아 줍니다.

In [ ]:
similar = image_db.similarity_search_by_image(image_uris[0], k=2)
ImageRetriever.plt_img_base64(similar[-1].page_content)